In [1]:
#!pip install pandas
#!pip install scikit-learn
#!pip install datasets
#!pip install transformers
#!pip install torch
#!pip install matplotlib
#!pip install seaborn
#!pip install accelerate>=0.26.0

In [2]:
import os
import ast
from ast import literal_eval

import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

import pickle

import torch
import torch.nn as nn

import numpy as np
from scipy.special import expit
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
model = "./baseline/medRoBERTa"
base_model = "CLTL/MedRoBERTa.nl"
model_name = "medRoBERTa"
data = "alldata"
lr_categories = 5e-6
lr_levels = 1e-5
lr_time = 2e-5
num_epochs = 3
batch_size = 16

In [4]:
def combine_df(input_dir):
    dfs = []

    for file in os.listdir(input_dir):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(input_dir, file))
            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [5]:
def load_df(input_path):
    file = os.path.basename(input_path)
    if file.endswith(".csv"):
        df = pd.read_csv(input_path)

    return df

In [6]:
category_names = [
    "B1300 Energy level",
    "B140 Attention functions",
    "B152 Emotional functions",
    "B440 Respiration functions",
    "B455 Exercise tolerance functions",
    "B530 Weight maintenance functions",
    "D450 Walking",
    "D550 Eating",
    "D840-859 Work and employment",
    "B280 Sensations of pain",
    "B134 Sleep functions",
    "D760 Family relationships",
    "B164 Higher-level cognitive functions",
    "D465 Moving around using equipment",
    "D410 Changing basic body position",
    "B230 Hearing functions",
    "D240 Handling stress and other psychological demands",
    "None"
]

In [ ]:
# Create special tokens
special_tokens = [f"[{name}]" for name in category_names] + ["[LEVELS]", "[TEXT]", "[HISTORY]"]

In [8]:
time_names = [
    "past",
    "present",
    "future",
    "None"
]

### Load the data

In [9]:
input_path_train = "conv_input_data/train"
train_df = combine_df(input_path_train)

In [10]:
input_path_dev = "conv_input_data/dev"
dev_df = combine_df(input_path_dev)

In [11]:
# Drop rows with NaN for text
train_df = train_df.dropna(subset=["text"])
dev_df = dev_df.dropna(subset=["text"])

## Categories

In [12]:
categories_train_df = train_df.copy()
categories_dev_df = dev_df.copy()

In [13]:
categories_train_df["categories"] = categories_train_df["categories"].apply(ast.literal_eval)
categories_dev_df["categories"] = categories_dev_df["categories"].apply(ast.literal_eval)

### Encode categories

In [14]:
invalid_rows = categories_dev_df[categories_dev_df["categories"].apply(lambda cats: any(cat not in category_names for cat in cats))]
print(f"There are {len(invalid_rows)} invalid rows in the development set.")

categories_train_df = categories_train_df.copy()
categories_train_df = categories_train_df[~categories_train_df["categories"].apply(
    lambda cats: any(cat not in category_names for cat in cats))]
categories_dev_df = categories_dev_df.copy()
categories_dev_df = categories_dev_df[~categories_dev_df["categories"].apply(
    lambda cats: any(cat not in category_names for cat in cats))]

There are 1562 invalid rows in the development set.


In [15]:
categories_mlb = MultiLabelBinarizer(classes=category_names)

categories_train_df["labels"] = list(categories_mlb.fit_transform(categories_train_df["categories"]))
categories_train_df["labels"] = categories_train_df["labels"].apply(lambda x: [float(v) for v in x])
categories_dev_df["labels"] = list(categories_mlb.transform(categories_dev_df["categories"]))
categories_dev_df["labels"] = categories_dev_df["labels"].apply(lambda x: [float(v) for v in x])

In [16]:
categories_train_dataset = Dataset.from_pandas(categories_train_df)
categories_dev_dataset = Dataset.from_pandas(categories_dev_df)

### Tokenise text with categories

In [17]:
# Load mBERT tokeniser
#categories_tokeniser = AutoTokenizer.from_pretrained(f"{model}_categories")
categories_tokeniser = AutoTokenizer.from_pretrained(f"./{data}/{model_name}_categories")

In [18]:
#categories_tokeniser.save_pretrained(f"./{data}/{model_name}_categories")

In [19]:
def tokenise_categories_function(examples):
    return categories_tokeniser(examples["text"], 
                     truncation=True, 
                     padding="max_length", 
                     max_length=512)

In [20]:
categories_train_dataset = categories_train_dataset.map(tokenise_categories_function, batched=True)
categories_dev_dataset = categories_dev_dataset.map(tokenise_categories_function, batched=True)

Map:   0%|          | 0/388674 [00:00<?, ? examples/s]

Map:   0%|          | 0/102820 [00:00<?, ? examples/s]

In [21]:
categories_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

categories_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

### Train the categories model

In [22]:
def categories_compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = torch.sigmoid(torch.tensor(logits))
    predictions = (probabilities > 0.5).int().numpy()

    return {"micro_f1": f1_score(labels, predictions, average="micro"),
           "macro_f1": f1_score(labels, predictions, average="macro"),
           "weighted_f1": f1_score(labels, predictions, average="weighted")}

In [23]:
"""categories_model = AutoModelForSequenceClassification.from_pretrained(f"{model}_categories", 
                                                                 num_labels=18,
                                                                 problem_type="multi_label_classification")"""
categories_model = AutoModelForSequenceClassification.from_pretrained(f"./{data}/{model_name}_categories")
categories_trainer = Trainer(model=categories_model)

In [24]:
"""categories_training_args = TrainingArguments(
    output_dir=f"./{data}/{model_name}/results_categories",
    learning_rate=lr_categories,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/{model_name}/logs_categories",
)
 
categories_trainer = Trainer(
    model=categories_model,
    args=categories_training_args,
    train_dataset=categories_train_dataset,
    eval_dataset=categories_dev_dataset,               
    compute_metrics=categories_compute_metrics
)

categories_trainer.train()"""

'categories_training_args = TrainingArguments(\n    output_dir=f"./{data}/{model_name}/results_categories",\n    learning_rate=lr_categories,\n    num_train_epochs=num_epochs,                    \n    per_device_train_batch_size=batch_size,\n    eval_strategy="epoch",\n    save_strategy="epoch",\n    logging_dir=f"./{data}/{model_name}/logs_categories",\n)\n \ncategories_trainer = Trainer(\n    model=categories_model,\n    args=categories_training_args,\n    train_dataset=categories_train_dataset,\n    eval_dataset=categories_dev_dataset,               \n    compute_metrics=categories_compute_metrics\n)\n\ncategories_trainer.train()'

### Saving categories model

In [25]:
# Save the trained categories model
#categories_model.save_pretrained(f"./{data}/{model_name}_categories")

## Levels

### Load the data

In [26]:
levels_train_df = train_df.copy()
levels_dev_df = dev_df.copy()

In [27]:
levels_train_df["categories"] = levels_train_df["categories"].apply(ast.literal_eval)
levels_dev_df["categories"] = levels_dev_df["categories"].apply(ast.literal_eval)
levels_train_df["levels"] = levels_train_df["levels"].apply(ast.literal_eval)
levels_dev_df["levels"] = levels_dev_df["levels"].apply(ast.literal_eval)

In [28]:
def duplicate_rows(df):
    expanded_rows = []
    
    for i, row in df.iterrows():
        categories = row["categories"]
        levels = row["levels"] 

        if len(categories) != len(levels):
            print(f"Skipping {row['turn_id']} from {row['conversation_id']}")
            continue
    
        if len(categories) == 0:
            categories = [None]
    
        for j, (cat, level) in enumerate(zip(categories, levels)):
            if level == "0":
                continue
            new_row = row.copy()
            new_row["full_turn_id"] = (f"{row['conversation_id']}_{row['turn_id']}")
            new_row["new_turn_id"] = f"{row['turn_id']}_{j}"
            new_row["category"] = str(cat)
            new_row["level"] = level
    
            expanded_rows.append(new_row)
    
    return pd.DataFrame(expanded_rows)

levels_train_df = duplicate_rows(levels_train_df)
levels_dev_df = duplicate_rows(levels_dev_df)

### Augment data 

In [29]:
levels_train_df["levels"] = levels_train_df["levels"].astype(str)
levels_dev_df["levels"] = levels_dev_df["levels"].astype(str)

In [30]:
levels_train_df["category"] = levels_train_df["category"].fillna("None")
levels_dev_df["category"] = levels_dev_df["category"].fillna("None")
levels_train_df["level"] = levels_train_df["level"].fillna("None")
levels_dev_df["level"] = levels_dev_df["level"].fillna("None")

In [31]:
def build_levels_input(df, history_size=3):
    turn_categories = {}
    for (conv_id, turn_id), turn_rows in df.groupby(["conversation_id", "full_turn_id"], sort=False):
        turn_categories[(conv_id, turn_id)] = "".join(f"[{cat}]" for cat in turn_rows["category"])

    turn_histories  = {}

    for conv_id, conv_df in df.groupby("conversation_id", sort=False):
        turn_ids = conv_df["full_turn_id"].drop_duplicates().tolist()

        for i, turn_id in enumerate(turn_ids):
            previous_turns = turn_ids[max(0, i-history_size):i]

            history = " ".join(turn_categories[(conv_id, prev_turn)] for prev_turn in previous_turns)

            turn_histories[(conv_id, turn_id)] = history

    print(turn_histories)
    return df.apply(lambda row:
                    f"[HISTORY] {turn_histories[(row['conversation_id'], row['full_turn_id'])]} "
                    f"[CURRENT] [{row['category']}] "
                    f"[TEXT] {row['text']}",
            axis=1)

In [32]:
# Combine the predicted categories with the original text for the level model input
#levels_train_df["combined_text"] = build_levels_input(levels_train_df)
#levels_dev_df["combined_text"] = build_levels_input(levels_dev_df)
levels_train_df = pd.read_csv("./results_intermission/level_train_df.csv")
levels_dev_df = pd.read_csv("./results_intermission/level_dev_df.csv")

In [33]:
#levels_train_df.to_csv(f"./results_intermission/level_train_df.csv")
#levels_dev_df.to_csv(f"./results_intermission/level_dev_df.csv")

### Encode levels

In [34]:
levels_train_df["labels"] = levels_train_df["level"].apply(lambda x: -1 if pd.isna(x) or x == "None" or x is None or x == "" else float(x))
levels_dev_df["labels"] = levels_dev_df["level"].apply(lambda x: -1 if pd.isna(x) or x == "None" or x is None or x == "" else float(x))

In [35]:
levels_train_df["level"] = levels_train_df["level"].astype(str)
levels_dev_df["level"] = levels_dev_df["level"].astype(str)

In [36]:
levels_train_dataset = Dataset.from_pandas(levels_train_df)
levels_dev_dataset = Dataset.from_pandas(levels_dev_df)

### Tokenise text with levels

In [37]:
# Load tokeniser
levels_tokeniser = AutoTokenizer.from_pretrained(f"{model}_levels")
#levels_tokeniser = AutoTokenizer.from_pretrained(f"./{data}/{model_name}_levels")

In [38]:
levels_tokeniser.add_special_tokens(
    {"additional_special_tokens": special_tokens}
)

1

In [ ]:
levels_tokeniser.save_pretrained(f"./{data}/{model_name}_levels")

('./alldata/medRoBERTa_levels_no0/tokenizer_config.json',
 './alldata/medRoBERTa_levels_no0/special_tokens_map.json',
 './alldata/medRoBERTa_levels_no0/vocab.json',
 './alldata/medRoBERTa_levels_no0/merges.txt',
 './alldata/medRoBERTa_levels_no0/added_tokens.json',
 './alldata/medRoBERTa_levels_no0/tokenizer.json')

In [ ]:
def tokenise_levels_function(examples):
    return levels_tokeniser(examples["combined_text"], 
                     truncation=True,  
                     padding="max_length", 
                     max_length=512)

In [41]:
levels_train_dataset = levels_train_dataset.map(tokenise_levels_function, batched=True)
levels_dev_dataset = levels_dev_dataset.map(tokenise_levels_function, batched=True)

Map:   0%|          | 0/400026 [00:00<?, ? examples/s]

Map:   0%|          | 0/105972 [00:00<?, ? examples/s]

In [42]:
levels_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

levels_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

### Train the levels model

In [43]:
class MaskedRegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits.squeeze()

        mask = labels != -1

        loss_fct = nn.MSELoss()

        loss = loss_fct(
                logits[mask],
                labels.float()[mask])

        return (loss, outputs) if return_outputs else loss

In [44]:
def levels_compute_metrics(eval_pred):
    preds, labels = eval_pred
    predictions = preds.squeeze()

    mask = (labels != -1) & ~np.isnan(labels)
    labels_filtered = labels[mask]
    predictions_filtered = predictions[mask]

    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = root_mean_squared_error(labels, predictions)

    return {"mae": mae,
            "mse": mse,
            "rmse": rmse
            }

In [45]:
levels_model = AutoModelForSequenceClassification.from_pretrained(f"{model}_levels", 
                                                              num_labels=1, #len(level_encoder.classes_),
                                                              problem_type="regression")
#levels_model = AutoModelForSequenceClassification.from_pretrained(f"./{data}/{model_name}_levels")
#levels_trainer = Trainer(model=levels_model)
levels_model.resize_token_embeddings(len(levels_tokeniser))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(52022, 768, padding_idx=1)

In [46]:
levels_training_args = TrainingArguments(
    output_dir=f"./{data}/{model_name}/results_levels",
    learning_rate=lr_levels,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/{model_name}/logs_levels",
)

levels_trainer = MaskedRegressionTrainer(
    model=levels_model,
    args=levels_training_args,
    train_dataset=levels_train_dataset,
    eval_dataset=levels_dev_dataset,   
    tokenizer=levels_tokeniser,
    compute_metrics=levels_compute_metrics
)

levels_trainer.train()#resume_from_checkpoint=True)

/tmp/P103572.1970816/ipykernel_2697365/2513314902.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MaskedRegressionTrainer.__init__`. Use `processing_class` instead.
  levels_trainer = MaskedRegressionTrainer(


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse
1,2.083600,nan,1.335098,2.163555,1.470903
2,2.318200,nan,1.417157,2.501670,1.581667
3,1.958000,nan,1.309487,2.203934,1.484565


TrainOutput(global_step=75006, training_loss=2.1206358116798247, metrics={'train_runtime': 12038.5015, 'train_samples_per_second': 99.687, 'train_steps_per_second': 6.231, 'total_flos': 3.157509540684534e+17, 'train_loss': 2.1206358116798247, 'epoch': 3.0})

### Saving the levels model

In [ ]:
# Save the trained levels model
levels_model.save_pretrained(f"./{data}/{model_name}_levels")